# nts-memory-v1: a follow-up made to stand on its own (Qwen3.5-4B + LoRA)

The memory adapter reads the **newest chat message with the last 4 messages before it** and answers one of:

| answer | when | example |
|---|---|---|
| `{"same": true}` | the message stands on its own (most messages) | "how big is Section 4?" |
| `{"message": "..."}` | it leans on the conversation: written in full | "and Friday?" → "when is Dr. Rao free on Friday?" |
| `{"ask": "..."}` | it leans on something the conversation doesn't have | "is that room free?" → "Which room do you mean?" |

What it writes goes the usual way in the app (lookup, rules, requests): no other adapter changes
(`src/agents/memory.py`, ROADMAP: memory v1).

**The set-up is v3's lookup notebook's** (Qwen3.5-4B in fp32 on the two T4s, LoRA r=16, loss on the answer only,
time budget, checkpoints, GGUF export at the server's llama.cpp build). What differs: a **fresh adapter**
(learning rate 2e-4), short rows (~600 tokens, `MAX_LEN` 1024), memory's row check and quick check.

**Input to add (Add Input):** `data/memory-train` (made by `python -m training.memory_data`; Gemini's rewordings
`distilled.jsonl` in the same folder are used when present). Expected time: ~4 h for all the rows.

First `TRIAL = True` (a few minutes: installs, loads, 20 steps, exports), then `TRIAL = False` and *Run all*.


# Training the memory adapter

Each row is one conversation from the app's own kinds: the earlier question is a lookup template, the reply is
the app's real answer, and the follow-up's full form is known exactly (the same template with one thing changed,
or a question about something the reply gave). Kinds: `change` ("and Friday?", "no, I meant Section 3", "aur
Friday ko?"), `two` ("and Friday at 3?"), `relative` ("an hour later?"), `chain` (a follow-up of a follow-up),
`refer` ("is she free then?", "the second one", "move my DS lecture there"), `list` (an item of another list),
`two_back`, `both` (a course with two teachers), `reply` (the answer to "Who do you mean?"), `request` (its
status, cancel, change), `whatif` ("ok do it"), `yes` (typed to "send this as a request?"), `each` ("and
separately?"), `same` (a message that needs nothing), `ask` (it leans on what isn't there), `mixed`. Askers:
teachers, class representatives and the timetable office.


## 1. Install the libraries

In [ ]:
%%capture
# True: also install flash-linear-attention, the fast kernels for Qwen3.5's linear-attention layers. Without it
# transformers says "falling back to its reference PyTorch implementation ... much slower". It may not support
# the T4: if this cell or training fails with it, set it back to False.
FAST_KERNELS = False
# Keep Kaggle's own CUDA build of torch: no other package may replace it.
!pip freeze | grep -iE "^(torch|torchvision|torchaudio)==" > /tmp/torch-pin.txt
# transformers v5 is needed for Qwen3.5; PEFT adds the LoRA adapter; TRL has the training loop (SFTTrainer).
!pip install --upgrade -c /tmp/torch-pin.txt "transformers>=5" peft trl accelerate datasets
# Kaggle ships an old torchao that PEFT refuses; nothing here uses it.
!pip uninstall -y -q torchao
if FAST_KERNELS:
    !pip install -c /tmp/torch-pin.txt flash-linear-attention

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # fewer out-of-memory errors from fragmentation

import torch, transformers, peft, trl

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
assert int(transformers.__version__.split(".")[0]) >= 5, "Qwen3.5 needs transformers v5: restart the kernel and run from here"

N_GPU = torch.cuda.device_count()
assert N_GPU > 0, "no GPU: set Settings > Accelerator > GPU T4 x2"
for i in range(N_GPU):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}, {torch.cuda.get_device_properties(i).total_memory / 2**30:.1f} GB")

## 2. Settings

Everything you might change is here.

In [ ]:
import glob, json, random, time
from pathlib import Path

SESSION_START = time.time()  # Kaggle stops a session after 12 hours, counted from here

MODEL = "Qwen/Qwen3.5-4B"   # the SAME base the team's adapters use: an adapter only works on its own base
ADAPTER = "nts-memory-v1"   # the adapter's name: the GGUF file and its name in adapters-lora.toml

TRIAL = False               # True: 20 steps only, to check the setup (and still export, to test the server)
PRECISION = "fp32"          # "fp32": slower but safe. "fp16": faster, but Qwen3.5's gradients overflowed in
                            # fp16 here (grad_norm nan and learning_rate 0 on every step: nothing was learned)
EPOCHS = 1                  # passes over the training rows (one is a lot for this task; raise it if time allows)
MAX_TRAIN_ROWS = None       # e.g. 800: train on a random part of the rows, for a shorter run (None: all)
TIME_BUDGET_H = 11.0        # stop training after this many hours of the session, saving the adapter, so the
                            # export below still runs inside Kaggle's 12 hours
SAVE_EVERY = 25             # also save a checkpoint every this many steps
LR = 2e-4                   # a fresh adapter (a later version continuing from this one: 5e-5)
MAX_LEN = 1024              # tokens per row; memory's rows are about 560 (checked below)
BATCH = 1                   # rows per step on the GPU (1 keeps memory low on a T4)
ACCUM = 16                  # steps added up before each update: an effective batch of 16 rows
LORA_R = 16                 # the adapter's size (rank); 16 is the team's choice
CHECK_ROWS = 120            # validation conversations to try at the end (answers are short)
SEED = 3407

# The data: the dataset added to this notebook, wherever Kaggle mounted it.
def _task(train_file):
    m = Path(train_file).parent / "manifest.json"
    return json.loads(m.read_text()).get("task") if m.exists() else None


found = [f for f in sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)) if _task(f) == "memory"]
assert found, "no memory train.jsonl: add the data/memory-train folder as a dataset (Add Input)"
DATA = Path(found[0]).parent

# The adapter to continue from: an earlier nts-memory's lora/ folder, added as an input (None: a fresh adapter,
# as for v1). Only a memory adapter: another task's (lookup's) is never a start for this one.
starts = [p for p in sorted(glob.glob("/kaggle/input/**/lora/adapter_config.json", recursive=True))
          if "nts-memory" in p and ADAPTER not in p]
START_ADAPTER = Path(starts[0]).parent if starts else None
# A run of THIS adapter cut short in an earlier session: its checkpoints, added as an input (None: step 0).
# Only this adapter's (its name is in the path): another run's checkpoints are not this one's.
ckpts = sorted((p for p in glob.glob("/kaggle/input/**/checkpoint-*/trainer_state.json", recursive=True)
                if ADAPTER in p), key=lambda p: int(Path(p).parent.name.split("-")[1]))
RESUME = str(Path(ckpts[-1]).parent) if ckpts else None
print("continue from:", START_ADAPTER or "a fresh adapter", "| resume:", RESUME or "no")
OUT = Path("/kaggle/working") / ADAPTER
OUT.mkdir(parents=True, exist_ok=True)
print("data:", DATA, "| output:", OUT)
random.seed(SEED)
torch.manual_seed(SEED)

## 3. Load the data and check every row

Each row is one chat, exactly as the app sends it to the model:

| message | content |
|---|---|
| `system` | the task and its rules (`SYSTEM` in `agents/memory.py`) |
| `user` | who asks, now, the week; the conversation (`<conversation>`: up to 4 messages with their times); the newest message (`<message>`) |
| `assistant` | the answer to learn: `{"same": true}`, `{"message": "..."}` or `{"ask": "..."}` |

A row that isn't like this is left out and shown, never trained on.


In [ ]:
from collections import Counter

ANSWERS = {"same", "message", "ask"}


def problem(row):
    # Why a row must not be trained on, or None when it is fine.
    msgs = row.get("messages")
    if not isinstance(msgs, list) or [m.get("role") for m in msgs] != ["system", "user", "assistant"]:
        return "messages are not system, user, assistant"
    if "<conversation>" not in msgs[1]["content"] or "<message>" not in msgs[1]["content"]:
        return "the user message has no <conversation> or <message>"
    try:
        answer = json.loads(msgs[2]["content"])
    except json.JSONDecodeError:
        return "the answer is not JSON"
    if not isinstance(answer, dict) or len(answer) != 1 or set(answer) - ANSWERS:
        return f"the answer is not exactly one of same, message, ask: {answer}"
    if "same" in answer and answer["same"] is not True or "same" not in answer and not str(next(iter(answer.values()))).strip():
        return f"an empty answer: {answer}"
    return None


def load(name):
    rows, dropped = [], []
    for n, line in enumerate(open(DATA / f"{name}.jsonl", encoding="utf-8"), 1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            dropped.append((n, "not a JSON line"))
            continue
        why = problem(row)
        if why:
            dropped.append((n, why))
        else:
            rows.append(row)
    print(f"{name}: {len(rows)} rows kept, {len(dropped)} left out")
    for n, why in dropped[:20]:
        print(f"   line {n}: {why}")
    return rows


train_rows, val_rows = load("train"), load("val")
if (DATA / "distilled.jsonl").exists():  # Gemini's rewordings of the follow-ups: training rows too
    train_rows += load("distilled")
assert train_rows and val_rows, "no usable rows"
if MAX_TRAIN_ROWS and len(train_rows) > MAX_TRAIN_ROWS:  # a shorter run: a random part, every kind still in it
    train_rows = random.Random(SEED).sample(train_rows, MAX_TRAIN_ROWS)
    print(f"training on {len(train_rows)} of the rows (MAX_TRAIN_ROWS)")


def answer_of(row):
    return next(iter(json.loads(row["messages"][2]["content"])))


print("\nanswers:    ", dict(Counter(answer_of(r) for r in train_rows).most_common()))
print("kinds:      ", dict(Counter(r.get("kind") for r in train_rows).most_common()))
print("departments:", dict(Counter(r.get("department") for r in train_rows).most_common()))
print("messages in the conversation:", dict(Counter(r["messages"][1]["content"].count("\nYou") +
                                                    r["messages"][1]["content"].count("\nAssistant")
                                                    for r in train_rows).most_common()))
assert {answer_of(r) for r in train_rows} == ANSWERS, "every answer must have training rows: make the data again"


In [ ]:
# One row, as the model will see it.
example = train_rows[0]["messages"]
print(example[1]["content"])
print("\nANSWER TO LEARN:", example[2]["content"])


## 4. Load the model and add the LoRA adapter

* **The base model** is loaded in `PRECISION` (fp32: about 17 GB) and split over the two T4s
  (`device_map="auto"`). Its own weights stay frozen.
* **The LoRA adapter** adds small trainable matrices to the attention and MLP layers of the *language*
  part only (Qwen3.5 also has a vision part, which we don't use). That is about 1% of the weights, and the
  only part that is trained.
* The adapter's own weights are always float32.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoTokenizer
from peft import LoraConfig, get_peft_model

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.chat_template is None:  # some releases keep the chat template with the processor
    from transformers import AutoProcessor
    tok.chat_template = AutoProcessor.from_pretrained(MODEL).chat_template

DTYPE = torch.float32 if PRECISION == "fp32" else torch.float16
model = AutoModelForImageTextToText.from_pretrained(
    MODEL, dtype=DTYPE, device_map="auto", attn_implementation="sdpa")
for i in range(N_GPU):
    print(f"GPU {i}: {torch.cuda.memory_allocated(i) / 2**30:.1f} GB of weights")
model.config.use_cache = False  # not needed while training; it would only use memory
print("placed on:", sorted({str(d) for d in model.hf_device_map.values()}))

if START_ADAPTER:  # an earlier nts-memory, trained further
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, str(START_ADAPTER), is_trainable=True)
    print("continuing from", START_ADAPTER)
else:
    model = get_peft_model(model, LoraConfig(
        r=LORA_R, lora_alpha=LORA_R, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
        # every attention and MLP projection, but not in the vision part ("visual")
        target_modules=r"^(?!.*visual).*\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$"))
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()  # the adapter in float32 (with fp16, the frozen base stays fp16)
model.print_trainable_parameters()

## 5. Train

Each row becomes a **prompt** (system + user, rendered with the chat template, thinking off, as the server
does) and a **completion** (the JSON answer + end of text). The loss is computed on the completion only: the
model learns to *answer*, not to repeat the prompt.

**Progress:** after every step a line with the loss, the seconds per step and the estimated finish, so you know
within minutes whether the run fits in Kaggle's 12 hours. A checkpoint is saved every `SAVE_EVERY` steps, and
training stops by itself at `TIME_BUDGET_H` with the adapter as far as it got, so a long run is never lost.

In [ ]:
from datasets import Dataset
from transformers import TrainerCallback
from trl import SFTConfig, SFTTrainer


class Clock(TrainerCallback):
    # One line per step: the loss, seconds per step, and when training will finish. Stops at TIME_BUDGET_H,
    # and stops a run that learns nothing (the gradients not a number on 8 steps in a row: fp16 overflow).
    def on_train_begin(self, args, state, control, **kwargs):
        self.start, self.nan_steps = time.time(), 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs:
            return
        grad = float(logs.get("grad_norm", 0.0))
        self.nan_steps = self.nan_steps + 1 if grad != grad or grad == float("inf") else 0
        if self.nan_steps >= 8:
            print("STOPPED: the gradients were not a number on 8 steps in a row, so nothing is learned. "
                  "Use PRECISION = 'fp32'.", flush=True)
            control.should_training_stop = True
        done, total = state.global_step, state.max_steps
        per_step = (time.time() - self.start) / max(done, 1)
        left_h = per_step * (total - done) / 3600
        print(f"step {done}/{total}  loss {logs['loss']:.4f}  {per_step:.0f} s/step  "
              f"about {left_h:.1f} h left", flush=True)

    def on_step_end(self, args, state, control, **kwargs):
        if not TRIAL and (time.time() - SESSION_START) / 3600 > TIME_BUDGET_H:
            print(f"time budget reached at step {state.global_step}: stopping, saved as a checkpoint (the next session resumes from it) and as the adapter below", flush=True)
            control.should_save = True
            control.should_training_stop = True



def prompt_of(messages):
    # system + user, as the server renders them (thinking off), ending where the answer starts
    return tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)


def to_example(row):
    return {"prompt": prompt_of(row["messages"]), "completion": row["messages"][2]["content"] + tok.eos_token}


train_ds = Dataset.from_list([to_example(r) for r in train_rows])
val_ds = Dataset.from_list([to_example(r) for r in val_rows])

lengths = [len(tok(e["prompt"] + e["completion"])["input_ids"]) for e in train_ds]
print(f"tokens per row: max {max(lengths)}, mean {sum(lengths) // len(lengths)}")
assert max(lengths) <= MAX_LEN, "a row is longer than MAX_LEN: raise MAX_LEN"

trainer = SFTTrainer(
    model=model,
    processing_class=tok,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    # the settings of the team's notebook (train_multigpu_kaggle), which train on two T4s with transformers v5
    args=SFTConfig(
        output_dir=str(OUT / "checkpoints"),
        max_length=MAX_LEN,
        completion_only_loss=True,          # learn the answer, not the prompt
        loss_type="nll",                     # TRL's default loss breaks when the model is split over two GPUs
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=ACCUM,
        num_train_epochs=EPOCHS,
        max_steps=20 if TRIAL else -1,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_steps=5,                      # a gentle start (transformers v5 has no warmup_ratio any more)
        optim="adamw_torch",                 # only the adapter trains, so the optimizer's memory is small
        fp16=PRECISION == "fp16",            # mixed precision only with fp16 (see PRECISION)
        bf16=False,                          # T4s have no bf16
        gradient_checkpointing=True,         # less memory, a little slower
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=1,                     # a progress line every step (see Clock)
        disable_tqdm=True,                   # the progress bar does not show in Kaggle's log; Clock does
        eval_strategy="no",                  # the quick check below does this, on unseen wordings
        save_strategy="steps",
        save_steps=SAVE_EVERY,
        save_total_limit=2,                  # the last two, so a session cut mid-save still has one
        report_to="none",
        seed=SEED,
    ),
    callbacks=[Clock()],
)
stats = trainer.train(resume_from_checkpoint=RESUME)  # RESUME: an earlier session's last checkpoint, or None
print(f"trained {trainer.state.global_step} steps in {stats.metrics['train_runtime'] / 60:.0f} min")

## 6. Quick check on the validation conversations

For each conversation: is the answer valid JSON, the right kind (same, message or ask), and for a rewrite, the
same words as expected (after lower case and punctuation). The real score, which accepts other wordings of the same
meaning, is `python -m evaluation.memory_set --local ...` on the test set, with the adapter served.


In [ ]:
import re

model.eval()
model.config.use_cache = True


def words(s):
    return re.sub(r"[^a-z0-9 ]+", " ", s.lower()).split()


rows = random.Random(SEED).sample(val_rows, min(CHECK_ROWS, len(val_rows)))
score, misses = Counter(), []
for row in rows:
    inputs = tok(prompt_of(row["messages"]), return_tensors="pt").to("cuda:0")  # the first layers are on GPU 0
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=160, do_sample=False, use_cache=True)
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    want = json.loads(row["messages"][2]["content"])
    kind = row.get("kind")
    score["conversations"] += 1
    score[f"asked {kind}"] += 1
    try:
        got = json.loads(text)
        score["valid JSON"] += 1
    except json.JSONDecodeError:
        got = None
    same_kind = isinstance(got, dict) and set(got) == set(want)
    exact = same_kind and ("message" not in want or words(got["message"]) == words(want["message"]))
    score["right kind"] += same_kind
    score["right"] += exact
    score[f"right {kind}"] += exact
    if not exact:
        misses.append((row["messages"][1]["content"].split("<conversation>")[1], want, text))

print(f"{score['right']}/{score['conversations']} exactly right, {score['right kind']} the right kind, "
      f"{score['valid JSON']} valid JSON")
for kind in sorted({r.get("kind") for r in rows}):
    print(f"   {kind:9} {score[f'right {kind}']}/{score[f'asked {kind}']}")
for conversation, want, text in misses[:12]:
    print(f"\n{conversation.strip()[-500:]}\n   want {json.dumps(want)}\n   got  {text}")


## 7. Save the adapter and convert it to GGUF

* `lora/`: the adapter as PEFT saves it (keep it: a later version can continue from it);
* `gguf/nts-memory-v1.gguf`: the same adapter for `llama-server`, made with llama.cpp's
  `convert_lora_to_gguf.py` **at the server's build (b11368)**, as the team's other adapters.


In [ ]:
model.save_pretrained(str(OUT / "lora"))
tok.save_pretrained(str(OUT / "lora"))
(OUT / "manifest.json").write_text(json.dumps({
    "adapter": ADAPTER, "base": MODEL, "task": "memory", "train_rows": len(train_rows), "val_rows": len(val_rows),
    "epochs": EPOCHS, "lr": LR, "lora_r": LORA_R, "trial": TRIAL, "version": 1,
    "start_adapter": str(START_ADAPTER) if START_ADAPTER else None, "resumed_from": RESUME,
    "check": {k: v for k, v in score.items() if k in ("conversations", "valid JSON", "right kind", "right")}}, indent=1))
print("saved", OUT / "lora")

In [ ]:
from huggingface_hub import snapshot_download

!git clone --depth 1 --branch b11368 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!pip install -q /tmp/llama.cpp/gguf-py sentencepiece
BASE_DIR = snapshot_download(MODEL)  # the base's config (already downloaded)
(OUT / "gguf").mkdir(exist_ok=True)
!python /tmp/llama.cpp/convert_lora_to_gguf.py {OUT}/lora --base {BASE_DIR} --outtype f16 --outfile {OUT}/gguf/{ADAPTER}.gguf
for f in sorted((OUT / "gguf").glob("*.gguf")):
    print(f, f"{f.stat().st_size / 2**20:.0f} MB")

## Next: serve it and measure it

1. Download `nts-memory-v1/gguf/nts-memory-v1.gguf` (and keep `lora/` and `manifest.json`) from the *Output* tab.
2. On the server, add it next to the others (same base, nothing else changes): `--lora ...,nts-memory-v1.gguf`.
3. In the project, `adapters-lora.toml` gets `nts-memory-v1 = ["memory"]`; the app then makes follow-ups stand on
   their own before reading them (`World.followup`).
4. Measure it on the **test set** (templates, wordings and names it never saw), against Gemini:
   `uv run python -m evaluation.memory_set --local http://<server>:8080/v1`.
